# Project 2: pretrained embeddings and recurrent encoders on ISEAR

| Section | Who |
| --- | --- |
| 0. Data and folds | Britt |
| 1. Embedding model, 2. What the embeddings capture |  |
| 3. Embed-and-pool | |
| 4. LSTM | |
| 5. Hyperparameter search and variance, 6. Error analysis | Together |

Run `python explore_data.py` once first: it creates `data/isear_clean.csv`.

In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold

import part1
import part2

results_folder = Path("results/project2")
results_folder.mkdir(parents=True, exist_ok=True)

## 0. Data and folds (same as project 1)
Same cleaned data (7,411 texts) and the same stratified 5 folds × 3 seeds as project 1. Because every text lands in the same fold as last week, our new systems are **paired** with the project 1 results, so we can compare them directly with the Bayesian test (Q3: did pretrained embeddings add to our design?).

In [5]:
LABELS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
outcome_space = part1.clslib.NamedClasses(LABELS)

df = pd.read_csv("data/isear_clean.csv")
pairs = list(zip(df["text"], df["label_id"]))  # (text, label) pairs, as in the tutorials
print(len(pairs), "texts")

def stratified_cv_split(pairs, k, seed):
    """Same split as project 1: every fold has the same mix of emotions."""
    labels = [y for _, y in pairs]
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    return {fold: {"training": [pairs[i] for i in tr], "validation": [pairs[i] for i in va]}
            for fold, (tr, va) in enumerate(skf.split(np.zeros(len(pairs)), labels))}

seeds = [0, 3, 5]
num_folds = 5
cv_sets = [stratified_cv_split(pairs, k=num_folds, seed=seed) for seed in seeds]

7411 texts


In [6]:
# Check: are these exactly the project 1 folds? (compares the validation texts of every seed and fold)
p1 = pd.read_csv("results/project1/predictions.csv")
p1 = p1[(p1["representation"] == "bow+bigrams") & (p1["architecture"] == "loglinear")]
row_of = {text: i for i, text in enumerate(df["text"])}

for seed, cv in zip(seeds, cv_sets):
    for fold, split in cv.items():
        ours = sorted(row_of[text] for text, _ in split["validation"])
        theirs = sorted(p1[(p1["seed"] == seed) & (p1["fold"] == fold)]["row"])
        assert ours == theirs, f"seed {seed} fold {fold} differs from project 1"
print("Folds are identical to project 1")

Folds are identical to project 1


## 1. Embedding model ()
Choose a pretrained embedding model (GloVe or fastText, see the tutorial's `download_pretrained`) and show it suits ISEAR:
- tokenise our texts the same way the embedding model defines a token (`part2.clslib.BasicTokenizer`; GloVe is lowercase);
- check vocabulary coverage: share of tokens and word types found, and the most frequent missing words.

**Produces:** `emb_model` (a `part2.clslib.PretrainedEmbeddings`), `tokenizer_for_pool` (no punctuation or stopwords) and `tokenizer_for_rnn` (keeps everything).

## 2. What the embeddings capture ()
Nearest neighbours and similarities of emotion words (e.g. ashamed / guilty, angry / disgusted), comparison between embedding models, and known limitations (e.g. opposites like happy / sad being close).

## 3. Embed-and-pool
`part2.experiment.cross_validation` with `enc_type='embed-and-pool'` on `cv_sets`. Design choices to test, e.g. frozen vs trainable embeddings, regressor size, dropout, learning rate. Compare with project 1 (paired folds).

**Produces:** per-fold accuracies saved in `results/project2/`.

## 4. LSTM
`part2.experiment.cross_validation` with `enc_type='lstm'`. Design choices to test, e.g. hidden size, frozen vs trainable embeddings, word dropout, training data size; watch for overfitting.

**Produces:** per-fold accuracies saved in `results/project2/`.

## 5. Hyperparameter search and variance (together)
How we structure the search, and the Bayesian comparison with a ROPE (tutorial `part1.analysis.BenavoliCorrelatedT`, `runs=3`).

Note: `part2.experiment.train` picks the best epoch by the loss on the validation fold, which is also the fold we report. To avoid that optimism, choose epochs and hyperparameters on a development split held out from the training fold.

## 6. Error analysis (together)
Which text characteristics predict where the systems differ, e.g. length, share of unknown words, negation, explicit emotion words, contrast words ("but"); tutorial BDA model (`part2.analysis.fit_bda_model`).